In [1]:
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
from face_position import Face_Position
from dofbot_utils.robot_controller import Robot_Controller
from dofbot_utils.fps import FPS
from dofbot_utils.dofbot_config import *

In [2]:
robot = Robot_Controller()
robot.move_init_pose()
fps = FPS()

In [3]:
face = Face_Position()
model = 'General'

In [4]:
button_layout = widgets.Layout(width='250px', height='50px', align_self='center')
output = widgets.Output()
# 退出控件 exit button
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
# 图像控件 Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
# 空间布局 spatial distribution
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))
# ['auto', 'flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'un

In [5]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
#     with output: print(model)
exit_button.on_click(exit_button_Callback)

In [6]:
def camera():
    global model
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0, cv.CAP_V4L2)
    capture.set(cv.CAP_PROP_FRAME_WIDTH, 640)
    capture.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
    while capture.isOpened():
        try:
            _, img = capture.read()
            fps.update_fps()
            img, pos = face.process(img)
            if pos is not None:
                # 在图像左下角显示坐标
                cv.putText(img, f"Pos: ({pos[0]}, {pos[1]})", 
               (10, img.shape[0]-20), 
               cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            else:
                # 显示未检测提示
                cv.putText(img, "No object detected!", 
               (10, img.shape[0]-20), 
               cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            if model == 'Exit':
                capture.release()
                break
            fps.show_fps(img)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

In [7]:
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()

[ WARN:0@0.741] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ WARN:0@0.741] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
